# Picking FDSN waveforms with RED-PAN-Motion

This notebook downloads the waveforms of one earthquake from public FDSN
services with ObsPy, runs the shipped checkpoints on them, and compares the
picks with the arrival times of the iasp91 model. It shows both ways to run a
checkpoint:

1. `REDPANPredictor`, the package's own interface, which takes NumPy arrays.
2. `RedpanSB90s` and `RedpanSB60s`, which wrap the checkpoints as SeisBench
   models that take ObsPy streams.

Requirements: the package with its SeisBench extra, which includes the
checkpoints,

    pip install "redpan_motion[seisbench] @ git+https://github.com/tso1257771/RED-PAN-Motion@v0.1.3"

and network access to the USGS and SCEDC FDSN services.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth, kilometers2degrees
from obspy.taup import TauPyModel

from redpan_motion import CH_Z, REDPANPredictor, highpass, picks_to_dataframe
from redpan_motion.integrations.seisbench import RedpanSB60s, RedpanSB90s

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.dpi": 100, "font.size": 9})
print("device:", DEVICE)

## 1. The earthquake

The origin comes from the USGS catalogue: the Mw 5.2 earthquake near Julian,
California, on 14 April 2025.

In [ ]:
catalog = Client("USGS").get_events(
    starttime=UTCDateTime("2025-04-14T17:00"), endtime=UTCDateTime("2025-04-14T17:15"),
    latitude=33.0, longitude=-116.6, maxradius=1.0, minmagnitude=5.0)
event = catalog[0]
origin, magnitude = event.preferred_origin(), event.preferred_magnitude()
ORIGIN_TIME = origin.time
EV_LAT, EV_LON, EV_DEPTH_KM = origin.latitude, origin.longitude, origin.depth / 1000
print(f"{ORIGIN_TIME}  {magnitude.magnitude_type} {magnitude.mag:.1f}  "
      f"lat {EV_LAT:.3f}  lon {EV_LON:.3f}  depth {EV_DEPTH_KM:.1f} km  "
      f"({event.event_descriptions[0].text})")

## 2. Stations and waveforms

Eight broadband stations of the Southern California Seismic Network (CI),
between 15 and 210 km from the epicentre, record at 100 Hz on HHE, HHN and HHZ.
The request covers 60 s before to 120 s after the origin. The table lists the
first P and S arrivals that iasp91 predicts, in seconds after the origin.

In [ ]:
scedc = Client("SCEDC")
STATIONS = ["CYP", "SDR", "IKP", "HMT2", "IMP", "KML", "PEM", "DAN"]
t1, t2 = ORIGIN_TIME - 60, ORIGIN_TIME + 120
st = scedc.get_waveforms("CI", ",".join(STATIONS), "*", "HH?", t1, t2)
inv = scedc.get_stations(network="CI", station=",".join(STATIONS), channel="HH?",
                         starttime=t1, endtime=t2, level="channel")
st.merge(fill_value=0)          # a gap, if any, becomes zeros
st.detrend("demean")
for tr in st:
    if tr.stats.sampling_rate != 100:
        tr.resample(100.0)      # the native interface expects 100 Hz
print(st.__str__(extended=True))

taup = TauPyModel("iasp91")


def first_arrivals(dist_km):
    arrivals = taup.get_travel_times(EV_DEPTH_KM, kilometers2degrees(dist_km), ["p", "P", "s", "S"])
    return (min(a.time for a in arrivals if a.name in ("p", "P")),
            min(a.time for a in arrivals if a.name in ("s", "S")))


rows = []
for sta in STATIONS:
    coords = inv.select(station=sta)[0][0]
    dist_km = gps2dist_azimuth(EV_LAT, EV_LON, coords.latitude, coords.longitude)[0] / 1000
    tp, ts = first_arrivals(dist_km)
    rows.append({"station": sta, "lat": coords.latitude, "lon": coords.longitude,
                 "dist_km": dist_km, "iasp91_P": tp, "iasp91_S": ts})
stations = pd.DataFrame(rows).set_index("station")
stations.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(EV_LON, EV_LAT, "r*", ms=14, label="epicentre")
ax.plot(stations["lon"], stations["lat"], "k^", ms=7, label="station")
for sta, row in stations.iterrows():
    ax.annotate(f"{sta} ({row['dist_km']:.0f} km)", (row["lon"], row["lat"]),
                textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set_xlabel("longitude (°)")
ax.set_ylabel("latitude (°)")
ax.set_aspect(1 / np.cos(np.radians(EV_LAT)))
ax.legend(loc="lower left")
ax.set_title("Epicentre and stations")
plt.show()

## 3. Native interface: `REDPANPredictor`

`REDPANPredictor.from_checkpoint("redpan_motion")` loads the shipped checkpoint
by name and rebuilds the network from its `config.json`. A path to a `.pt` file
works too. `sp_adaptive_thresholds=True` selects the checkpoint's detection
thresholds that vary with S-P time, the same table the SeisBench wrappers use
by default.

The model input is a `(3, T)` array at 100 Hz with rows E, N and Z, high-passed
at 1 Hz. The polarity head of `redpan_motion` reads the raw vertical separately
through `z_raw`, so that the sign of the first motion is kept. `picks_to_dataframe`
pairs a P and an S pick inside each detection and reads the first-motion
polarity at the P pick. Its amplitude columns need the instrument response
(`level="response"` in `get_stations`) and are left empty here with
`amplitude=False`.

By default the predictor slides the 90 s window along the record in 10 s steps,
combines the overlapping outputs, and then reruns one window that starts 5 s
before each detection.

In [ ]:
COMPONENT_ROW = {"E": 0, "1": 0, "N": 1, "2": 1, "Z": 2, "3": 2}


def to_array(st3):
    """One instrument's three components as a (3, T) float32 array, rows E, N, Z."""
    t0 = max(tr.stats.starttime for tr in st3)
    t1 = min(tr.stats.endtime for tr in st3)
    st3 = st3.slice(t0, t1)
    n = min(tr.stats.npts for tr in st3)
    x = np.zeros((3, n), dtype=np.float32)
    for tr in st3:
        x[COMPONENT_ROW[tr.stats.channel[-1]]] = tr.data[:n]
    return x, t0


def pick_native(checkpoint):
    """Run one checkpoint on every station; return the picks and the raw outputs."""
    predictor = REDPANPredictor.from_checkpoint(checkpoint, device=DEVICE, sp_adaptive_thresholds=True)
    picks, outputs = [], {}
    for sta in STATIONS:
        st3 = st.select(station=sta)
        raw, t0 = to_array(st3)
        dt = st3[0].stats.delta
        picker, detector, polarity = predictor.predict_arrays(
            highpass(raw, 1.0, dt), z_raw=raw[CH_Z])
        outputs[sta] = {"raw": raw, "t0": t0, "picker": picker,
                        "detector": detector, "polarity": polarity}
        picks.append(picks_to_dataframe(
            picker, detector, polarity, t0, f"CI.{sta}.{st3[0].stats.location}.HH",
            dt=dt, amplitude=False, sp_thresholds=predictor.sp_thresholds))
    picks = pd.concat(picks, ignore_index=True)
    picks["station"] = picks["id"].str.split(".").str[1]
    picks["phase"] = picks["type"].str.upper()
    picks["t_after_origin"] = [UTCDateTime(t) - ORIGIN_TIME for t in picks["timestamp"]]
    return picks, outputs


native, outputs = pick_native("redpan_motion")
native[["station", "phase", "t_after_origin", "prob", "polarity", "polarity_prob"]].round(2)

Model input and output at one station. The P, S and detection probabilities
come from the picker and detector heads. The polarity panel shows the up and
down probabilities of the polarity head. They are read only at the P pick.

In [ ]:
sta = "SDR"
o = outputs[sta]
t = (o["t0"] - ORIGIN_TIME) + np.arange(o["raw"].shape[1]) * 0.01
x = highpass(o["raw"], 1.0, 0.01)
fig, ax = plt.subplots(5, 1, figsize=(9, 7), sharex=True)
for i, comp in enumerate("ENZ"):
    ax[i].plot(t, x[i] / np.abs(x).max(), "k", lw=0.5)
    ax[i].set_ylabel(f"HH{comp}")
ax[3].plot(t, o["picker"][:, 0], "C0", label="P")
ax[3].plot(t, o["picker"][:, 1], "C1", label="S")
ax[3].plot(t, o["detector"][:, 0], color="0.6", label="detection")
ax[3].set_ylabel("probability")
ax[4].plot(t, o["polarity"][:, 1], "C2", label="up")
ax[4].plot(t, o["polarity"][:, 2], "C3", label="down")
ax[4].set_ylabel("polarity")
for a in ax[3:]:
    a.set_ylim(0, 1.05)
    a.legend(loc="upper right", ncol=3, fontsize=8)
for _, pk in native[native["station"] == sta].iterrows():
    for a in ax:
        a.axvline(pk["t_after_origin"], color="C0" if pk["phase"] == "P" else "C1", ls="--", lw=1)
ax[-1].set_xlim(-10, 50)
ax[-1].set_xlabel("time after origin (s)")
ax[0].set_title(f"CI.{sta}, {stations.loc[sta, 'dist_km']:.0f} km: redpan_motion input "
                "(1 Hz highpass) and output; dashed lines are the picks")
plt.show()

Record section of the vertical components, with the picks of `redpan_motion`
and the iasp91 arrival curves.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for sta in STATIONS:
    o = outputs[sta]
    t = (o["t0"] - ORIGIN_TIME) + np.arange(o["raw"].shape[1]) * 0.01
    z = highpass(o["raw"], 1.0, 0.01)[CH_Z]
    d = stations.loc[sta, "dist_km"]
    ax.plot(t, d + 10 * z / np.abs(z).max(), "k", lw=0.4)
    ax.text(-9, d + 2, sta, fontsize=8)
grid = np.linspace(5, 225, 60)
tp_grid, ts_grid = np.array([first_arrivals(d) for d in grid]).T
ax.plot(tp_grid, grid, "C0", lw=1, alpha=0.5, label="iasp91 P")
ax.plot(ts_grid, grid, "C1", lw=1, alpha=0.5, label="iasp91 S")
for phase, color in (("P", "C0"), ("S", "C1")):
    pk = native[native["phase"] == phase]
    ax.scatter(pk["t_after_origin"], stations.loc[pk["station"], "dist_km"],
               marker="|", s=300, lw=2, color=color, label=f"redpan_motion {phase}", zorder=3)
ax.set_xlim(-10, 80)
ax.set_xlabel("time after origin (s)")
ax.set_ylabel("epicentral distance (km)")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Vertical components (1 Hz highpass) with picks")
plt.show()

The raw vertical around each P pick, with the first motion the polarity head
reports and its probability.

In [ ]:
p_picks = native[native["phase"] == "P"]
fig, axes = plt.subplots(2, 4, figsize=(10, 4.2), sharex=True, sharey=True)
for ax, (_, pk) in zip(axes.flat, p_picks.iterrows()):
    o = outputs[pk["station"]]
    i = round((UTCDateTime(pk["timestamp"]) - o["t0"]) / 0.01)
    seg = o["raw"][CH_Z, i - 50:i + 51].astype(float)
    seg -= seg[:40].mean()          # baseline from 0.5 to 0.1 s before the pick
    ax.plot(np.arange(-50, 51) * 0.01, seg / np.abs(seg).max(), "k", lw=0.8)
    ax.axvline(0, color="C0", ls="--", lw=1)
    ax.axhline(0, color="0.7", lw=0.5)
    call = "up" if pk["polarity"] > 0 else "down"
    ax.set_title(f"{pk['station']}: {call} ({pk['polarity_prob']:.2f})")
    ax.set_yticks([])
fig.supxlabel("time from the P pick (s)")
fig.suptitle("Raw HHZ at the P pick and the redpan_motion polarity call")
plt.tight_layout()
plt.show()

## 4. SeisBench interface

`from_redpan_checkpoint` loads a shipped checkpoint, by name, or a checkpoint
directory into its SeisBench class:
`RedpanSB90s` for `redpan_motion` and `edge_rp90`, `RedpanSB60s` for
`redpan_60s`. The models take an ObsPy stream at any sampling rate, resample it
to 100 Hz, apply the same 1 Hz highpass, and run on windows of 90 s (60 s for
`redpan_60s`). `RedpanSB90s` steps its window by 75 s and discards the outer
7.5 s of each window, the smallest overlap that leaves no gap. `annotate`
returns the probability traces and `classify` returns the picks, with the first
motion as SeisBench's `up` or `down`.

In [ ]:
sb_models = {
    "redpan_motion": RedpanSB90s.from_redpan_checkpoint("redpan_motion", device=DEVICE),
    "edge_rp90": RedpanSB90s.from_redpan_checkpoint("edge_rp90", device=DEVICE),
    "redpan_60s": RedpanSB60s.from_redpan_checkpoint("redpan_60s", device=DEVICE),
}
rows = []
for name, model in sb_models.items():
    for p in model.classify(st).picks:
        rows.append({"checkpoint": name, "station": p.trace_id.split(".")[1], "phase": p.phase,
                     "t_after_origin": p.peak_time - ORIGIN_TIME, "prob": p.peak_value,
                     "polarity": p.polarity})
sb = pd.DataFrame(rows)
sb[sb["checkpoint"] == "redpan_motion"].round(2)

The annotations of the three checkpoints at one station.

In [ ]:
sta = "SDR"
st_sta = st.select(station=sta)
fig, ax = plt.subplots(4, 1, figsize=(9, 6.5), sharex=True)
z = st_sta.copy().filter("highpass", freq=1.0).select(component="Z")[0]
ax[0].plot(z.times(reftime=ORIGIN_TIME), z.data / np.abs(z.data).max(), "k", lw=0.5)
ax[0].set_ylabel("HHZ")
for a, (name, model) in zip(ax[1:], sb_models.items()):
    ann = model.annotate(st_sta)
    for label, color in (("P", "C0"), ("S", "C1"), ("Detection", "0.6")):
        tr = ann.select(channel=f"{model.name}_{label}")[0]
        a.plot(tr.times(reftime=ORIGIN_TIME), tr.data, color=color, label=label)
    a.set_ylim(0, 1.05)
    a.set_ylabel(name)
ax[1].legend(loc="upper right", ncol=3, fontsize=8)
ax[-1].set_xlim(-10, 50)
ax[-1].set_xlabel("time after origin (s)")
ax[0].set_title(f"CI.{sta}: SeisBench annotations of the three checkpoints")
plt.show()

## 5. The two interfaces compared

Pick times of `redpan_motion` through both interfaces, in seconds after the
origin, beside the iasp91 predictions. An empty cell means no P and S pair was
found at that station.

In [ ]:
def by_station(df, phase, column="t_after_origin"):
    return df[df["phase"] == phase].groupby("station")[column].first()


mo = sb[sb["checkpoint"] == "redpan_motion"]
table = pd.DataFrame({
    "dist_km": stations["dist_km"],
    "iasp91_P": stations["iasp91_P"], "native_P": by_station(native, "P"), "seisbench_P": by_station(mo, "P"),
    "iasp91_S": stations["iasp91_S"], "native_S": by_station(native, "S"), "seisbench_S": by_station(mo, "S"),
    "native_polarity": by_station(native, "P", "polarity").map({1.0: "up", -1.0: "down"}),
    "seisbench_polarity": by_station(mo, "P", "polarity"),
}).reindex(STATIONS)
table.round(2)

Stations with a P and S pair, for each checkpoint and interface.

In [ ]:
counts = {}
for name in sb_models:
    nat = native if name == "redpan_motion" else pick_native(name)[0]
    counts[name] = {"native": nat.loc[nat["phase"] == "P", "station"].nunique(),
                    "seisbench": sb.loc[(sb["checkpoint"] == name) & (sb["phase"] == "P"), "station"].nunique()}
pd.DataFrame(counts).T.assign(stations=len(STATIONS))

The SeisBench windows start at the first sample of the record, so the record
start decides where their edges fall. The same data cut to start 30 s before
the origin:

In [ ]:
later = st.slice(ORIGIN_TIME - 30, ORIGIN_TIME + 120)
pd.DataFrame({
    name: {"seisbench, record from origin - 60 s": counts[name]["seisbench"],
           "seisbench, record from origin - 30 s": len(
               {p.trace_id.split(".")[1] for p in model.classify(later).picks if p.phase == "P"})}
    for name, model in sb_models.items()}).T

## Summary

- The native `redpan_motion` P picks lie within 0.8 s of the iasp91 times at
  all eight stations. The S picks differ from iasp91 by up to 3.4 s, at DAN, the
  farthest station. These differences combine pick error with the misfit of a
  global velocity model to the crust of southern California.
- Where both interfaces return a pair, the `redpan_motion` pick times differ
  by at most 0.1 s and the polarity calls agree.
- The SeisBench interface found fewer pairs for `redpan_motion` and
  `edge_rp90`. `REDPANPredictor` sees every arrival in several overlapping
  windows. `RedpanSB90s` sees it in one or two, and an arrival near a window
  edge is seen with little signal on one side. Which stations lose their pair
  depends on where the window edges fall, as the record that starts 30 s before
  the origin shows.